In [1]:
import sys
sys.path.append('..')
from search.retrieval import *

corpus, inverted_index, bm25_data = load_indices()
faiss_index, faiss_doc_ids, model = load_faiss()
graph, synonyms = load_graph()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/katharinazeilnhofer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
No sentence-transformers model found with name T-Systems-onsite/cross-en-de-roberta-sentence-transformer. Creating a new one with mean pooling.


Loaded: 12106 documents, 63177 tokens


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: T-Systems-onsite/cross-en-de-roberta-sentence-transformer
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS loaded: 12106 vectors
Graph loaded: 29613 nodes, 70378 edges


In [2]:
query = "Kartoffelsuppe"

print("=== BM25 ===")
for doc_id, title, score in bm25_search(query, corpus, bm25_data):
    print(f"  [{score:.2f}] {title}")

print("\n=== Semantic Search (FAISS) ===")
for doc_id, title, dist in faiss_search(query, corpus, faiss_index, faiss_doc_ids, model):
    print(f"  [{dist:.4f}] {title}")

print("\n=== Graph Search ===")
for doc_id, title in graph_search(["Kartoffel", "Zwiebel", "Speck"], graph, corpus):
    print(f"  {title}")

=== BM25 ===
  [10.15] Kartoffelsuppe I
  [8.72] Zutat:Dampfnudel
  [8.53] Belgische Kartoffelsuppe
  [7.39] Zutat:Mousseron
  [7.24] Zutat:Asia Wok-Gewürzmischung

=== Semantic Search (FAISS) ===
  [37.8229] Zutat:Nova
  [37.8229] Zutat:Selma
  [37.9644] Zutat:Christa
  [37.9644] Zutat:Colette
  [37.9644] Zutat:Conny

=== Graph Search ===
  Friesischer verzierter Grünkohl
  Geschmortes Rinderherz
  Deftiger Bohneneintopf
  Kürbissuppe spezial
  Grundrezept Sauerkraut


In [3]:
# without graph filter 
print("=== Hybrid: 'Suppe' ===")
for _, title, score in hybrid_search("Suppe", corpus, bm25_data, faiss_index, faiss_doc_ids, model):
    print(f"  [{score:.4f}] {title}")

# with graph filter
print("\n=== Hybrid: 'Suppe' + Filter 'Kartoffel' ===")
for _, title, score in hybrid_search("Suppe", corpus, bm25_data, faiss_index, faiss_doc_ids, model, 
                                      graph=graph, filter_zutaten=["Kartoffel"]):
    print(f"  [{score:.4f}] {title}")

=== Hybrid: 'Suppe' ===
  [0.8549] Klare Suppe mit Ei
  [0.8522] Brunnenkressesuppe mit Käse
  [0.8454] Spargelrahmsuppe
  [0.8446] Caldo verde
  [0.8435] Grießsuppe

=== Hybrid: 'Suppe' + Filter 'Kartoffel' ===
  [0.8446] Caldo verde
  [0.8322] Kartoffelcremesuppe
  [0.8202] Kräutersuppe mit Ei
  [0.8185] Fischsuppe mit Gemüse
  [0.7957] Klare, leichte Kartoffelsuppe
